## Upload MNIST to S3 as a Single Object

### Objective
Download the MNIST `.npz` archive if needed, wrap the whole file as one LAILA entry, upload it to S3, and store the manifest as another S3-backed LAILA entry for the training notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

### Objective
Import the notebook dependencies, configure paths, and define the reusable constants for this example.

In [ ]:
import hashlib
import json
import sys
import urllib.request
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path("/home/ubuntu/laila")
PYTHON_ROOT = PROJECT_ROOT.parent
if str(PYTHON_ROOT) not in sys.path:
    sys.path.append(str(PYTHON_ROOT))

import laila
laila.read_args(PROJECT_ROOT / "vault" / "dev_secrets.toml")
from laila.pool import S3Pool

OFFICIAL_MNIST_NPZ_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
POOL_NICKNAME = "mnist_s3_pool"
MANIFEST_NICKNAME = "mnist_s3_manifest"
ARCHIVE_PATH = PROJECT_ROOT / "examples" / "training" / "mnist" / "mnist.npz"

### Objective
Define helpers for downloading the archive and constructing an S3 pool from `laila.args`.

In [ ]:
def ensure_mnist_npz(npz_path: Path) -> Path:
    # Reuse the local file if it already exists.
    if npz_path.exists():
        return npz_path

    # Otherwise download the official compact MNIST archive.
    print(f"Downloading MNIST archive to {npz_path} ...")
    urllib.request.urlretrieve(OFFICIAL_MNIST_NPZ_URL, npz_path)
    return npz_path


def build_s3_pool() -> S3Pool:
    required = ["AWS_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
    missing = [name for name in required if getattr(laila.args, name, None) is None]
    if missing:
        raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )


def manifest_entry_id_from_nickname(manifest_nickname: str) -> str:
    # Nicknames deterministically map to entry IDs, so we can re-derive this later.
    return laila.constant(data=np.zeros(1, dtype=np.uint8), nickname=manifest_nickname).global_id

### Objective
Load the MNIST archive bytes, compute a hash for integrity checking, and convert the full file into one `uint8` array payload.

In [ ]:
archive_path = ensure_mnist_npz(ARCHIVE_PATH)
archive_bytes = archive_path.read_bytes()
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()

# Store the whole file as a single uint8 payload rather than multipart chunks.
archive_array = np.frombuffer(archive_bytes, dtype=np.uint8).copy()

print("Archive:", archive_path)
print("Archive size (bytes):", len(archive_bytes))
print("SHA256:", archive_sha256)

### Objective
Register the S3 pool and wrap the archive as one deterministic LAILA entry.

In [ ]:
s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)

# Single-object upload: one entry for the full MNIST archive.
archive_entry = laila.constant(
    data=archive_array,
    nickname=f"mnist-npz-{archive_sha256[:12]}",
)

print("Entry id:", archive_entry.global_id)

### Objective
Upload the archive as one object and wait on the single future returned by `laila.memorize(...)`.

In [ ]:
# Single entry upload: LAILA returns a regular future instead of a GroupFuture.
upload_future = laila.memorize(archive_entry, pool_nickname=POOL_NICKNAME)
print("Upload future type:", type(upload_future).__name__)

upload_future.wait()
print("Upload complete.")

### Objective
Create a manifest that records the archive hash, pool nickname, and the single entry ID needed for the training notebook, then memorize that manifest onto S3 as its own LAILA entry.

In [ ]:
manifest_entry_id = manifest_entry_id_from_nickname(MANIFEST_NICKNAME)
manifest = {
    "archive_path": str(archive_path),
    "archive_name": archive_path.name,
    "archive_size_bytes": len(archive_bytes),
    "archive_sha256": archive_sha256,
    "pool_nickname": POOL_NICKNAME,
    "manifest_nickname": MANIFEST_NICKNAME,
    "manifest_entry_id": manifest_entry_id,
    "entry_id": archive_entry.global_id,
}

manifest_entry = laila.constant(data=manifest, nickname=MANIFEST_NICKNAME)
manifest_future = laila.memorize(manifest_entry, pool_nickname=POOL_NICKNAME)
print("Manifest future type:", type(manifest_future).__name__)
manifest_future.wait()

print("Manifest upload complete.")
print("Manifest entry id:", manifest_entry.global_id)
print(json.dumps(manifest, indent=2))